In [3]:
import os, time, random, warnings
from pathlib import Path


import numpy as np
import pandas as pd
import psutil
import torch


from codecarbon import EmissionsTracker
from bstabdiff import BSTabDiff


from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils import check_random_state


warnings.filterwarnings("ignore")
os.environ["CODECARBON_LOG_LEVEL"]="critical"


GLOBAL_SEED=42
CV_SEED=0
CSV_PATH="coloncancer_encoded.csv"
LABEL_COL="label"
N_SYN=200
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR=Path("bstabdiff_carbon_outputs")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)


random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)


def select_default_M(n,m):
    rho=m/n
    kappa=n*m
    if m<=64: return m
    if rho<10: return 64
    if kappa<2.5e5: M=32
    elif kappa<1e6: M=64
    elif kappa<3e6: M=128
    else: M=192
    return min(M,m)


def make_bstabdiff(M,seed):
    return BSTabDiff(
        M=M,
        prior_type="diffusion",
        device=DEVICE,
        random_state=seed,
        prior_epochs=10_000,
        prior_batch=128,
        prior_lr=1e-3,
        use_ema=True,
        ema_decay=0.999,
        permute_features=False,
        use_gobs=False,
        use_gobs_fc=False,
        verbose_every=0,
    )


def make_lr():
    return Pipeline([
        ("imp",SimpleImputer(strategy="median")),
        ("sc",StandardScaler(with_mean=True,with_std=True)),
        ("clf",LogisticRegression(
            max_iter=10_000,
            class_weight="balanced",
            solver="lbfgs",
            n_jobs=None
        ))
    ])


# ============================================================
# DATA
# ============================================================
df=pd.read_csv(CSV_PATH)
if LABEL_COL not in df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found.")


y_raw=df[LABEL_COL].to_numpy()
X_df=df.drop(columns=[LABEL_COL]).select_dtypes(include=[np.number])
X=X_df.to_numpy(dtype=np.float32)
X=np.nan_to_num(X,nan=0.0,posinf=0.0,neginf=0.0)


y,_=pd.factorize(y_raw)
y=y.astype(np.int64)


n,m=X.shape
M_full=select_default_M(n,m)


print("="*80)
print("BSTabDiff — Colon — Carbon Tracking + 5x5 TSTR Evaluation")
print("="*80)
print(f"Shape               : {X.shape}")
print(f"Classes             : {np.unique(y).tolist()}")
print(f"Device              : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
print(f"Default M           : {M_full}")
print(f"Synthetic samples   : {N_SYN}")
print(f"Permutation         : OFF")
print(f"GO-BS               : OFF")
print(f"GO-BS-FC            : OFF")
print(f"TSTR CV             : 5 folds x 5 repeats = 25 runs")
print("="*80)


# ============================================================
# PART A — BSTabDiff GENERATIVE MODELING / ENERGY-CARBON
#
# This section measures ONE full-data BSTabDiff run:
#     real data -> BSTabDiff fit -> synthetic data
#
# CodeCarbon covers ONLY this section.
# Logistic Regression and the 5x5 CV below are NOT included.
# ============================================================
model=make_bstabdiff(M_full,GLOBAL_SEED)
process=psutil.Process(os.getpid())
cpu_mem_before_gb=process.memory_info().rss/(1024**3)


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()


tracker=EmissionsTracker(
    project_name="BSTabDiff_Colon_Default_NoPermutation",
    output_dir=str(OUTPUT_DIR),
    output_file="emissions.csv",
    measure_power_secs=1,
    save_to_file=True,
    log_level="critical",
)


tracker.start()
gen_start=time.perf_counter()


try:
    model.fit(X,y)
    X_syn_full,R_syn_full,y_syn_full=model.sample(n_samples=N_SYN,return_mask=True)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
finally:
    emissions_kg=tracker.stop()


gen_runtime_sec=time.perf_counter()-gen_start
cpu_mem_after_gb=process.memory_info().rss/(1024**3)
peak_gpu_gb=torch.cuda.max_memory_allocated()/(1024**3) if torch.cuda.is_available() else np.nan


carbon_data=tracker.final_emissions_data


def cc_value(name,default=np.nan):
    if carbon_data is None: return default
    value=getattr(carbon_data,name,default)
    return default if value is None else value


energy_kwh=cc_value("energy_consumed")
cpu_energy_kwh=cc_value("cpu_energy")
gpu_energy_kwh=cc_value("gpu_energy")
ram_energy_kwh=cc_value("ram_energy")
if emissions_kg is None:
    emissions_kg=cc_value("emissions")


# ============================================================
# PART B — DOWNSTREAM TSTR EVALUATION
#
# 5x5 repeated stratified CV = 25 independent held-out runs.
#
# For EACH fold:
#   GENERATIVE STEP:
#       real training fold -> BSTabDiff -> synthetic training data
#
#   DOWNSTREAM STEP:
#       synthetic training data -> Logistic Regression
#       Logistic Regression -> untouched real test fold
#
# CodeCarbon is OFF for this entire CV evaluation section.
# ============================================================
cv=RepeatedStratifiedKFold(n_splits=5,n_repeats=5,random_state=CV_SEED)
rng=check_random_state(CV_SEED)


rows=[]
acc_scores=[]
auc_scores=[]
cv_start=time.perf_counter()


for run,(train_idx,test_idx) in enumerate(cv.split(X,y),start=1):
    run_seed=int(rng.randint(0,2**31-1))
    X_train,X_test=X[train_idx],X[test_idx]
    y_train,y_test=y[train_idx],y[test_idx]
    M_fold=select_default_M(len(X_train),m)


    # --------------------------------------------------------
    # GENERATIVE MODEL FOR THIS EVALUATION FOLD
    # Fit ONLY on the real training fold.
    # --------------------------------------------------------
    fold_model=make_bstabdiff(M_fold,run_seed)
    fold_model.fit(X_train,y_train)
    X_syn,_,y_syn=fold_model.sample(n_samples=N_SYN,return_mask=True)


    if len(np.unique(y_syn))<2:
        raise RuntimeError(f"Run {run}: synthetic data contains fewer than two classes.")


    # --------------------------------------------------------
    # DOWNSTREAM MODEL
    # Logistic Regression is NOT part of BSTabDiff.
    # Train on synthetic data, test on held-out real data.
    # --------------------------------------------------------
    lr=make_lr()
    lr.fit(X_syn,y_syn)


    y_pred=lr.predict(X_test)
    y_prob=lr.predict_proba(X_test)


    acc=accuracy_score(y_test,y_pred)
    if len(np.unique(y))==2:
        auc=roc_auc_score(y_test,y_prob[:,1])
    else:
        auc=roc_auc_score(y_test,y_prob,multi_class="ovr",average="macro")


    acc_scores.append(acc)
    auc_scores.append(auc)


    repeat=(run-1)//5+1
    fold=(run-1)%5+1


    rows.append({
        "run":run,
        "repeat":repeat,
        "fold":fold,
        "seed":run_seed,
        "n_train":len(train_idx),
        "n_test":len(test_idx),
        "M":M_fold,
        "tstr_accuracy":acc,
        "tstr_auc":auc,
    })


    print(
        f"Run {run:02d}/25 | Repeat {repeat} Fold {fold} | "
        f"Acc={acc:.4f} | AUC={auc:.4f}"
    )


cv_runtime_sec=time.perf_counter()-cv_start


acc_scores=np.asarray(acc_scores,dtype=float)
auc_scores=np.asarray(auc_scores,dtype=float)


tstr_acc_mean=acc_scores.mean()
tstr_acc_std=acc_scores.std(ddof=1)
tstr_auc_mean=auc_scores.mean()
tstr_auc_std=auc_scores.std(ddof=1)


# ============================================================
# SAVE RESULTS
# ============================================================
synthetic_path=OUTPUT_DIR/"colon_bstabdiff_synthetic_full.csv"
mask_path=OUTPUT_DIR/"colon_bstabdiff_missingness_mask_full.csv"
folds_path=OUTPUT_DIR/"bstabdiff_colon_tstr_5x5_folds.csv"
summary_path=OUTPUT_DIR/"bstabdiff_colon_carbon_tstr_summary.csv"


syn_df=pd.DataFrame(X_syn_full,columns=X_df.columns)
syn_df[LABEL_COL]=y_syn_full
syn_df.to_csv(synthetic_path,index=False)


pd.DataFrame(R_syn_full,columns=X_df.columns).to_csv(mask_path,index=False)
pd.DataFrame(rows).to_csv(folds_path,index=False)


summary=pd.DataFrame([{
    "dataset":"Colon",
    "n":n,
    "m":m,
    "M_full":M_full,
    "n_synthetic":N_SYN,
    "device":DEVICE,
    "prior_type":"diffusion",
    "prior_epochs":10_000,
    "prior_batch":128,
    "prior_lr":1e-3,
    "use_ema":True,
    "ema_decay":0.999,
    "permutation":False,
    "use_gobs":False,
    "use_gobs_fc":False,


    # One full-data BSTabDiff run only
    "generation_runtime_sec":gen_runtime_sec,
    "energy_kwh":energy_kwh,
    "cpu_energy_kwh":cpu_energy_kwh,
    "gpu_energy_kwh":gpu_energy_kwh,
    "ram_energy_kwh":ram_energy_kwh,
    "emissions_kg_co2e":emissions_kg,
    "peak_gpu_memory_gb":peak_gpu_gb,
    "cpu_memory_before_gb":cpu_mem_before_gb,
    "cpu_memory_after_gb":cpu_mem_after_gb,


    # Separate downstream evaluation
    "downstream_model":"LogisticRegression",
    "cv_folds":5,
    "cv_repeats":5,
    "cv_runs":25,
    "cv_evaluation_runtime_sec":cv_runtime_sec,
    "tstr_accuracy_mean":tstr_acc_mean,
    "tstr_accuracy_std":tstr_acc_std,
    "tstr_auc_mean":tstr_auc_mean,
    "tstr_auc_std":tstr_auc_std,
}])


summary.to_csv(summary_path,index=False)


# ============================================================
# FINAL REPORT
# ============================================================
print("\n"+"="*80)
print("PART A — BSTabDiff GENERATIVE MODELING")
print("="*80)
print(f"Generation runtime      : {gen_runtime_sec:.2f} s")
print(f"Total energy            : {energy_kwh:.8f} kWh")
print(f"CPU energy              : {cpu_energy_kwh:.8f} kWh")
print(f"GPU energy              : {gpu_energy_kwh:.8f} kWh")
print(f"RAM energy              : {ram_energy_kwh:.8f} kWh")
print(f"CO2e emissions          : {emissions_kg:.8f} kg")
print(f"Peak GPU memory         : {peak_gpu_gb:.4f} GiB")
print(f"CPU memory before       : {cpu_mem_before_gb:.4f} GiB")
print(f"CPU memory after        : {cpu_mem_after_gb:.4f} GiB")


print("\n"+"="*80)
print("PART B — DOWNSTREAM TSTR: LOGISTIC REGRESSION, 5x5 CV")
print("="*80)
print("Training               : BSTabDiff synthetic data")
print("Testing                : Held-out real Colon data")
print("Runs                   : 25")
print(f"TSTR Accuracy          : {tstr_acc_mean:.4f} ± {tstr_acc_std:.4f}")
print(f"TSTR ROC-AUC           : {tstr_auc_mean:.4f} ± {tstr_auc_std:.4f}")
print(f"CV evaluation runtime  : {cv_runtime_sec:.2f} s")
print("Included in energy     : NO")
print("="*80)


print("\nSaved:")
print(f"  {OUTPUT_DIR/'emissions.csv'}")
print(f"  {summary_path}")
print(f"  {folds_path}")
print(f"  {synthetic_path}")
print(f"  {mask_path}")


display(summary)

BSTabDiff — Colon — Carbon Tracking + 5x5 TSTR Evaluation
Shape               : (62, 2000)
Classes             : [0, 1]
Device              : cuda
GPU                 : NVIDIA RTX A6000
Default M           : 32
Synthetic samples   : 200
Permutation         : OFF
GO-BS               : OFF
GO-BS-FC            : OFF
TSTR CV             : 5 folds x 5 repeats = 25 runs
Run 01/25 | Repeat 1 Fold 1 | Acc=0.5385 | AUC=0.8250
Run 02/25 | Repeat 1 Fold 2 | Acc=0.7692 | AUC=0.8000
Run 03/25 | Repeat 1 Fold 3 | Acc=0.8333 | AUC=0.8750
Run 04/25 | Repeat 1 Fold 4 | Acc=1.0000 | AUC=1.0000
Run 05/25 | Repeat 1 Fold 5 | Acc=0.9167 | AUC=1.0000
Run 06/25 | Repeat 2 Fold 1 | Acc=0.8462 | AUC=0.9000
Run 07/25 | Repeat 2 Fold 2 | Acc=0.9231 | AUC=0.8750
Run 08/25 | Repeat 2 Fold 3 | Acc=0.6667 | AUC=0.6250
Run 09/25 | Repeat 2 Fold 4 | Acc=0.9167 | AUC=0.8750
Run 10/25 | Repeat 2 Fold 5 | Acc=0.9167 | AUC=1.0000
Run 11/25 | Repeat 3 Fold 1 | Acc=0.6923 | AUC=0.7750
Run 12/25 | Repeat 3 Fold 2 | Acc=0.923

,dataset,n,m,M_full,n_synthetic,device,prior_type,prior_epochs,prior_batch,prior_lr,...,cpu_memory_after_gb,downstream_model,cv_folds,cv_repeats,cv_runs,cv_evaluation_runtime_sec,tstr_accuracy_mean,tstr_accuracy_std,tstr_auc_mean,tstr_auc_std
0,Colon,62,2000,32,200,cuda,diffusion,10000,128,0.001,...,1.261581,LogisticRegression,5,5,25,861.649013,0.82641,0.123774,0.88475,0.098625
